# Question 2: Clustering from Scratch
**AI 2002 — Assignment 7**

Implements K-Means and K-Medoids **entirely from scratch** using only `numpy`, `pandas`, and `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

np.random.seed(42)

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('clustering_dataset.csv')
print('Shape:', df.shape)
print(df.describe())
X = df.values

plt.figure(figsize=(6,5))
plt.scatter(X[:,0], X[:,1], alpha=0.6, edgecolors='k', linewidths=0.3)
plt.title('Raw Data')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.tight_layout()
plt.show()

## 2. Helper Functions

In [ ]:
def euclidean_distance(a, b):
    """Euclidean distance between two points."""
    return np.sqrt(np.sum((a - b) ** 2))

def assign_clusters(X, centers):
    """Assign each point to the nearest center. Returns array of cluster indices."""
    labels = np.array([np.argmin([euclidean_distance(x, c) for c in centers]) for x in X])
    return labels

def compute_sse(X, labels, centers):
    """Compute SSE per cluster and total SSE."""
    k = len(centers)
    sse_per = []
    for i in range(k):
        pts = X[labels == i]
        if len(pts) > 0:
            sse_per.append(np.sum((pts - centers[i]) ** 2))
        else:
            sse_per.append(0.0)
    return sse_per, sum(sse_per)

## 3. K-Means from Scratch

In [ ]:
def kmeans_scratch(X, k, max_iter=300, tol=1e-6, verbose=False):
    """
    K-Means clustering from scratch.
    Returns: labels, centroids, total_sse, n_iterations
    """
    n = len(X)
    # Random initialization: pick k distinct points as initial centroids
    idx = np.random.choice(n, k, replace=False)
    centroids = X[idx].copy()

    labels = np.zeros(n, dtype=int)

    for iteration in range(1, max_iter + 1):
        # Assignment step
        new_labels = assign_clusters(X, centroids)

        # Compute SSE
        sse_per, total_sse = compute_sse(X, new_labels, centroids)

        if verbose:
            print(f"  Iteration {iteration}")
            for ci in range(k):
                count = np.sum(new_labels == ci)
                print(f"    Cluster {ci+1}: {count} points | SSE = {sse_per[ci]:.4f}")
            print(f"    Total SSE = {total_sse:.4f}")

        # Update step: recompute centroids
        new_centroids = np.array([
            X[new_labels == i].mean(axis=0) if np.sum(new_labels == i) > 0 else centroids[i]
            for i in range(k)
        ])

        # Convergence check
        shift = np.max([euclidean_distance(centroids[i], new_centroids[i]) for i in range(k)])
        centroids = new_centroids
        labels = new_labels

        if shift < tol:
            if verbose:
                print(f"  Converged at iteration {iteration} (shift={shift:.2e})")
            break

    sse_per, total_sse = compute_sse(X, labels, centroids)
    return labels, centroids, total_sse, iteration

## 4. Elbow Curve (K = 1 to 12)

In [ ]:
sse_list = []
K_range = range(1, 13)

for k in K_range:
    # Run multiple times and take best SSE (to reduce sensitivity to random init)
    best_sse = float('inf')
    for _ in range(10):
        _, _, sse, _ = kmeans_scratch(X, k)
        if sse < best_sse:
            best_sse = sse
    sse_list.append(best_sse)
    print(f"K={k:2d}  SSE={best_sse:.4f}")

# ---- Elbow detection using second-difference method ----
sse_arr = np.array(sse_list)
# Normalise to [0,1] for stable curvature
sse_norm = (sse_arr - sse_arr.min()) / (sse_arr.max() - sse_arr.min())
k_vals   = np.array(list(K_range), dtype=float)
k_norm   = (k_vals - k_vals.min()) / (k_vals.max() - k_vals.min())

# Distance of each point from the line joining first and last point
p1 = np.array([k_norm[0],  sse_norm[0]])
p2 = np.array([k_norm[-1], sse_norm[-1]])
dists = []
for i in range(len(k_norm)):
    p = np.array([k_norm[i], sse_norm[i]])
    d = np.abs(np.cross(p2 - p1, p1 - p)) / np.linalg.norm(p2 - p1)
    dists.append(d)
optimal_k = list(K_range)[np.argmax(dists)]
print(f"\nOptimal K (elbow) = {optimal_k}")

# ---- Plot ----
plt.figure(figsize=(9, 5))
plt.plot(list(K_range), sse_list, 'bo-', linewidth=2, markersize=7, label='SSE')
plt.scatter([optimal_k], [sse_list[optimal_k-1]],
            color='red', s=200, zorder=5, marker='*',
            label=f'Elbow at K={optimal_k}')
plt.annotate(f'  Optimal K={optimal_k}\n  SSE={sse_list[optimal_k-1]:.2f}',
             xy=(optimal_k, sse_list[optimal_k-1]),
             xytext=(optimal_k + 0.5, sse_list[optimal_k-1] + sse_list[0]*0.04),
             fontsize=11, color='red',
             arrowprops=dict(arrowstyle='->', color='red'))
plt.xlabel('Number of Clusters (K)', fontsize=13)
plt.ylabel('SSE / WCSS', fontsize=13)
plt.title('Elbow Curve — K-Means (From Scratch)', fontsize=14)
plt.xticks(list(K_range))
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('elbow_curve_scratch.png', dpi=150)
plt.show()
print(f"Optimal K selected: {optimal_k}")

## 5. K-Means Verbose Run at Optimal K

In [ ]:
print(f"\n{'='*55}")
print(f" K-Means (scratch) — K = {optimal_k}")
print(f"{'='*55}")

# Best run across 10 seeds
best_sse = float('inf')
best_seed = 42
for seed in range(10):
    np.random.seed(seed)
    _, _, s, _ = kmeans_scratch(X, optimal_k)
    if s < best_sse:
        best_sse = s
        best_seed = seed

np.random.seed(best_seed)

t0 = time.time()
km_labels, km_centroids, km_sse, km_iters = kmeans_scratch(X, optimal_k, verbose=True)
km_time = time.time() - t0

print(f"\nK-Means converged in {km_iters} iterations")
print(f"Final Total SSE = {km_sse:.4f}")
print(f"Wall-clock time : {km_time:.6f} seconds")

## 6. K-Medoids from Scratch

In [ ]:
def kmedoids_scratch(X, k, max_iter=300, tol=1e-6, verbose=False):
    """
    K-Medoids (PAM-style) clustering from scratch.
    Returns: labels, medoid_indices, medoid_points, total_sse, n_iterations
    """
    n = len(X)
    # Precompute full distance matrix for efficiency
    dist_mat = np.array([[euclidean_distance(X[i], X[j]) for j in range(n)] for i in range(n)])

    # Random initialisation: pick k distinct data points as medoids
    medoid_idx = list(np.random.choice(n, k, replace=False))

    for iteration in range(1, max_iter + 1):
        # Assignment: each point → nearest medoid
        labels = np.argmin(dist_mat[:, medoid_idx], axis=1)

        if verbose:
            sse_per, total_sse = compute_sse(X, labels, X[medoid_idx])
            print(f"  Iteration {iteration}")
            print(f"  Medoids (indices): {medoid_idx}")
            print(f"  Medoids (coords) : {[X[m].tolist() for m in medoid_idx]}")
            for ci in range(k):
                count = np.sum(labels == ci)
                print(f"    Cluster {ci+1}: {count} points | SSE = {sse_per[ci]:.4f}")
            print(f"    Total SSE = {total_sse:.4f}")

        # Update: for each cluster, find the point that minimises within-cluster cost
        new_medoid_idx = []
        for ci in range(k):
            cluster_pts = np.where(labels == ci)[0]
            if len(cluster_pts) == 0:
                new_medoid_idx.append(medoid_idx[ci])
                continue
            # Cost = sum of distances from candidate to all cluster members
            costs = dist_mat[np.ix_(cluster_pts, cluster_pts)].sum(axis=1)
            best_local = cluster_pts[np.argmin(costs)]
            new_medoid_idx.append(int(best_local))

        if new_medoid_idx == medoid_idx:
            if verbose:
                print(f"  Converged at iteration {iteration}")
            break

        medoid_idx = new_medoid_idx

    labels = np.argmin(dist_mat[:, medoid_idx], axis=1)
    sse_per, total_sse = compute_sse(X, labels, X[medoid_idx])
    return labels, medoid_idx, X[medoid_idx], total_sse, iteration

## 7. K-Medoids Verbose Run at Optimal K

In [ ]:
print(f"\n{'='*55}")
print(f" K-Medoids (scratch) — K = {optimal_k}")
print(f"{'='*55}")

# Best run across 10 seeds
best_sse_med = float('inf')
best_seed_med = 42
for seed in range(10):
    np.random.seed(seed)
    _, _, _, s, _ = kmedoids_scratch(X, optimal_k)
    if s < best_sse_med:
        best_sse_med = s
        best_seed_med = seed

np.random.seed(best_seed_med)

t1 = time.time()
kmed_labels, kmed_medoid_idx, kmed_medoids, kmed_sse, kmed_iters = \
    kmedoids_scratch(X, optimal_k, verbose=True)
kmed_time = time.time() - t1

print(f"\nK-Medoids converged in {kmed_iters} iterations")
print(f"Final Total SSE  = {kmed_sse:.4f}")
print(f"Wall-clock time  : {kmed_time:.6f} seconds")

## 8. Side-by-Side Final Cluster Visualisation

In [ ]:
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- K-Means ---
ax = axes[0]
for i in range(optimal_k):
    pts = X[km_labels == i]
    ax.scatter(pts[:,0], pts[:,1], color=colors[i], alpha=0.65,
               label=f'Cluster {i+1} (n={len(pts)})', edgecolors='k', linewidths=0.2)
for i, c in enumerate(km_centroids):
    ax.scatter(*c, color='black', s=220, marker='X', zorder=6,
               edgecolors='white', linewidths=1.2)
ax.scatter([], [], color='black', s=150, marker='X', label='Centroid')
ax.set_title(f'K-Means (Scratch) — K={optimal_k}\nSSE={km_sse:.2f} | {km_iters} iters | {km_time:.4f}s',
             fontsize=12)
ax.set_xlabel('Feature_1'); ax.set_ylabel('Feature_2')
ax.legend(fontsize=8, loc='best')
ax.grid(True, linestyle='--', alpha=0.4)

# --- K-Medoids ---
ax = axes[1]
for i in range(optimal_k):
    pts = X[kmed_labels == i]
    ax.scatter(pts[:,0], pts[:,1], color=colors[i], alpha=0.65,
               label=f'Cluster {i+1} (n={len(pts)})', edgecolors='k', linewidths=0.2)
for i, m in enumerate(kmed_medoids):
    ax.scatter(*m, color='black', s=220, marker='D', zorder=6,
               edgecolors='white', linewidths=1.2)
ax.scatter([], [], color='black', s=150, marker='D', label='Medoid')
ax.set_title(f'K-Medoids (Scratch) — K={optimal_k}\nSSE={kmed_sse:.2f} | {kmed_iters} iters | {kmed_time:.4f}s',
             fontsize=12)
ax.set_xlabel('Feature_1'); ax.set_ylabel('Feature_2')
ax.legend(fontsize=8, loc='best')
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Final Clustering Results — From Scratch', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('clusters_scratch.png', dpi=150)
plt.show()

## 9. Timing Summary

In [ ]:
print("\n" + "="*55)
print(" EXECUTION TIME SUMMARY (from scratch)")
print("="*55)
print(f" K-Means  : {km_time:.6f} seconds")
print(f" K-Medoids: {kmed_time:.6f} seconds")
print(f" K-Medoids is ~{kmed_time/km_time:.1f}x slower than K-Means")
print("="*55)

## 10. Comparative Analysis

### (a) Iterations to Converge
K-Medoids typically requires **more iterations** than K-Means because its update step is more conservative — it only swaps a medoid if doing so strictly reduces the within-cluster cost, whereas K-Means recomputes centroids algebraically in one step. Additionally, the PAM-style search over all cluster members to find a new medoid is computationally expensive per iteration, compounding the total wall-clock time.

### (b) Cluster Shapes and Sizes
Both algorithms produce similarly shaped clusters on this 2-D dataset. However, because K-Medoids restricts centroids to *actual data points*, its cluster boundaries can differ slightly from K-Means — particularly near the edges of dense blobs. Cluster sizes may vary: K-Medoids can produce slightly more unequal partitions when the medoid naturally falls off-centre in an irregular cluster.

### (c) Suitability for This Dataset
K-Means is **more suitable** here. The dataset appears to consist of roughly spherical, well-separated clusters with no obvious outliers — exactly the scenario where K-Means excels. K-Medoids offers robustness to outliers at the cost of significantly higher execution time (O(n²) distance matrix vs O(nk) for K-Means). Given comparable SSE values and the relatively clean structure of the data, K-Means is the preferable choice.
